In [22]:
# Teste SQLite3
import sqlite3

# Verificar versão do SQLite
print(f"SQLite version: {sqlite3.sqlite_version}")
print(f"Python SQLite3 bindings: {sqlite3.version}")

# Criar conexão de teste
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.execute('SELECT 1')
result = cursor.fetchone()
print(f"Conexão com SQLite OK: {result}")
conn.close()

print("✓ SQLite3 pronto para usar!")

SQLite version: 3.45.1
Python SQLite3 bindings: 2.6.0
Conexão com SQLite OK: (1,)
✓ SQLite3 pronto para usar!


/tmp/ipykernel_12645/2467619134.py:6: DeprecationWarning: version is deprecated and will be removed in Python 3.14
  print(f"Python SQLite3 bindings: {sqlite3.version}")


In [23]:
# Exemplo do que fazer
import pandas as pd

orders = pd.read_csv("/workspaces/olist-bi-dashboard/data/raw/olist_orders_dataset.csv")

# 1. Converter datas
date_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# 2. Calcular prazo de entrega
orders['dias_entrega'] = (
    orders['order_delivered_customer_date'] - 
    orders['order_purchase_timestamp']
).dt.days

# 3. Flag de atraso: 1 = atrasou, 0 = no prazo
# Compara a data real de entrega com a data prometida
orders['entrega_atrasada'] = (
    orders['order_delivered_customer_date'] > 
    orders['order_estimated_delivery_date']
)

### 🚩 Flag de Atraso — `entrega_atrasada`

Coluna derivada que marca **1** se a entrega atrasou e **0** se foi no prazo.

- `1` → cliente recebeu depois da data prometida  
- `0` → entrega dentro do prazo

Facilita as consultas SQL e os filtros no Power BI sem precisar comparar datas toda vez.

In [24]:
orders.to_csv("/workspaces/olist-bi-dashboard/data/processed/orders_clean.csv", index=False)

In [25]:
import sqlite3

In [27]:
# Receita total por mês usando SQL
cconn = sqlite3.connect("/workspaces/olist-bi-dashboard/data/processed/orders_clean.csv")

query_receita_mensal = """
SELECT
    strftime('%Y-%m', order_purchase_timestamp) AS mes,
    ROUND(SUM(price + freight_value), 2) AS receita_total,
    COUNT(DISTINCT o.order_id) AS total_pedidos
FROM orders o
JOIN order_items i ON o.order_id = i.order_id
WHERE o.order_status = 'delivered'
GROUP BY mes
ORDER BY mes
"""

receita_mensal = pd.read_sql(query_receita_mensal, conn)
receita_mensal.tail(10)


ProgrammingError: Cannot operate on a closed database.